# Exercise — Give the Model Grader More Context

**Task:** give the model grader more context on what a *good* solution looks like, so its scores are grounded in explicit criteria rather than its own assumptions.

**Two changes** on top of the code-based-grading pipeline:

1. **Dataset** — the generation prompt now asks for a `"solution_criteria"` field on each test case ("key criteria for evaluating the solution").
2. **Model grader** — `grade_by_model` injects those criteria into a `<criteria>` block, telling Claude exactly what to grade against.

> **Adaptations (unchanged from the section):** Haiku 4.5 (prefill), `get_text()`, explicit dataset path. We regenerate `dataset.json` so each task carries `solution_criteria`.

## Setup

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

import json
import ast
import re
from statistics import mean
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5-20251001"


def get_text(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return get_text(message)


print(f"Ready. Model: {model}")

Ready. Model: claude-haiku-4-5-20251001


## Step 1 — dataset with `solution_criteria`

Add a `"solution_criteria"` field to the example output so every generated task ships with an explicit rubric for the grader.

In [2]:
DATASET_PATH = "01-building-with-the-claude-api/02-prompt-evaluation/dataset.json"


def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
  {
    "task": "Description of task",
    "format": "json" or "python" or "regex",
    "solution_criteria": "Key criteria for evaluating the solution"
  },
  ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


dataset = generate_dataset()
with open(DATASET_PATH, "w") as f:
    json.dump(dataset, f, indent=2)

dataset

[{'task': "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message. The log format is: '[TIMESTAMP] [LEVEL] Message text'",
  'format': 'regex',
  'solution_criteria': 'Regex should correctly capture three groups: timestamp (ISO 8601 format), log level (ERROR, WARN, INFO, DEBUG), and message text. Should handle variable-length messages and special characters.'},
 {'task': "Write a Python function that takes an AWS S3 bucket name and file key, then returns the S3 URI in the format 's3://bucket-name/key'. The function should validate that the bucket name is not empty and follows AWS naming conventions (lowercase, hyphens allowed).",
  'format': 'python',
  'solution_criteria': 'Function should return valid S3 URIs, validate bucket names (3-63 characters, lowercase alphanumeric and hyphens), and handle edge cases like leading/trailing slashes in keys.'},
 {'task': 'Create a JSON CloudFormation template snippet that defines an AWS IAM role with a trust relations

## Step 2 — feed `solution_criteria` into the model grader

The grader now gets a `<criteria>` block. This grounds the score in the task's own rubric instead of the model's generic assumptions about quality.

In [3]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Criteria you should use to evaluate the solution:
<criteria>
{test_case["solution_criteria"]}
</criteria>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
"""

    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

## The rest of the pipeline (unchanged)

Prompt-under-test, syntax validators + dispatcher, combined score, and the eval loop — same as the code-based-grading lesson.

In [4]:
def run_prompt(test_case):
    prompt = f"""
Please solve the following task:

{test_case["task"]}

* Respond only with Python, JSON, or a plain Regex
* Do not add any comments or commentary or explanation
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output


def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0


def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0


def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)


def run_test_case(test_case):
    """Calls run_prompt, then grades the result with both graders"""
    output = run_prompt(test_case)

    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning,
    }


def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")

    return results

## Run it

The grader is now anchored to each task's `solution_criteria`, so its reasoning should reference those specific requirements rather than generic "could be more robust" boilerplate.

In [5]:
with open(DATASET_PATH, "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)
print(json.dumps(results, indent=2))

Average score: 7.0
[
  {
    "output": "\nimport re\nimport json\n\ndef parse_cloudwatch_log(log_entry):\n    pattern = r'\\[([^\\]]+)\\]\\s+\\[([^\\]]+)\\]\\s+(.+)'\n    match = re.match(pattern, log_entry)\n    if match:\n        return {\n            \"timestamp\": match.group(1),\n            \"level\": match.group(2),\n            \"message\": match.group(3)\n        }\n    return None\n\n# Example usage\nlog_entry = \"[2024-01-15T10:30:45Z] [ERROR] Database connection failed\"\nresult = parse_cloudwatch_log(log_entry)\nprint(json.dumps(result, indent=2))\n",
    "test_case": {
      "task": "Parse an AWS CloudWatch log entry and extract the timestamp, log level, and message. The log format is: '[TIMESTAMP] [LEVEL] Message text'",
      "format": "regex",
      "solution_criteria": "Regex should correctly capture three groups: timestamp (ISO 8601 format), log level (ERROR, WARN, INFO, DEBUG), and message text. Should handle variable-length messages and special characters."
    },
